# Woche 3 · ML-Intuition: Lernen heißt Optimieren
**AI Engineering: von LLMs zu Agenten** · linkit e.V. am KIT

Letzte Woche hat ein Sprachmodell geantwortet. Heute klären wir, was es heißt, dass ein Modell etwas **lernt**. Die Antwort ist überraschend schlicht: Ein Modell ist eine Funktion mit Stellschrauben, und Lernen heißt, die Stellschrauben so zu drehen, dass ein Fehlermaß klein wird.

**Lernziele**
- Ein Modell als parametrisierte Funktion $f_\theta(x)$ beschreiben
- Loss-Funktionen (MSE, Cross-Entropy) begründen
- Gradient Descent herleiten und implementieren
- Ein neuronales Netz als Verkettung von Funktionen verstehen, Backpropagation als Kettenregel
- Overfitting an Trainings- und Validierungsdaten erkennen

**Aufbau:** Teil A Regression von Hand (NumPy) · Teil B dasselbe mit PyTorch · Teil C ein neuronales Netz klassifiziert Störungen · Teil D Ausblick auf Sprachmodelle

Pflicht: TODO 1–6. TODO 7–8 sind das Ziel, Bonus für Schnelle. Heute ist kein API-Key nötig.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# NumPy und PyTorch sind in Colab vorinstalliert. Wir installieren sie bewusst nicht neu.
print("NumPy", np.__version__, "| PyTorch", torch.__version__)

def check(name, test):
    """Selbsttest: zeigt ✅ oder ❌. Fehler im Test gelten als 'noch nicht gelöst'."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

---
# Teil A · Lineare Regression: Lernen von Hand

## A1 · Modell und Loss

**Szenario:** Die Instandhaltung will den Energieverbrauch einer Schicht aus der Stückzahl vorhersagen. Wir haben 40 Schichten der Nordwerk GmbH (synthetisch).
$x$ ist die Stückzahl in Hundert Teilen, $y$ der Energieverbrauch der Schicht in MWh.

Unser **Modell** ist eine Gerade mit zwei Parametern $\theta = (w, b)$:

$$\hat{y} = f_\theta(x) = w\,x + b$$

Der **Loss** misst, wie falsch das Modell liegt. Wir nehmen die mittlere quadratische Abweichung (MSE):

$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} \big(y_i - \hat{y}_i\big)^2$$

In [ ]:
rng = np.random.default_rng(1)
x = rng.uniform(0.5, 4.0, 40)                      # Stückzahl in Hundert Teilen
y = 0.6 + 0.35 * x + rng.normal(0, 0.08, 40)       # Energie in MWh (Grundlast + Anteil pro Teil + Rauschen)

plt.figure(figsize=(6, 3.5))
plt.scatter(x, y)
plt.xlabel("Stückzahl (in Hundert)"); plt.ylabel("Energie (MWh)"); plt.title("40 Schichten der Nordwerk GmbH")
plt.show()

### TODO 1 · Vorhersage und MSE
Schreibt `vorhersage(x, w, b)` und `mse(y, y_dach)`. Beide sollen mit ganzen NumPy-Arrays funktionieren (keine Schleife nötig).

In [ ]:
def vorhersage(x, w, b):
    # TODO 1a
    pass

def mse(y, y_dach):
    # TODO 1b
    pass

# Die Einstiegsfrage: fünf Punkte und zwei Kandidaten-Geraden
xs = np.array([1, 2, 3, 4, 5.0]); ys = np.array([2.1, 3.9, 6.2, 7.8, 10.1])
print("Gerade y = 2x:          MSE =", mse(ys, vorhersage(xs, 2.0, 0.0)) if vorhersage(xs, 2, 0) is not None else None)
print("Gerade y = 1.99x + 0.05: MSE =", mse(ys, vorhersage(xs, 1.99, 0.05)) if vorhersage(xs, 2, 0) is not None else None)

In [ ]:
check("TODO 1a vorhersage", lambda: np.allclose(vorhersage(np.array([0.0, 1.0, 2.0]), 2.0, 1.0), [1.0, 3.0, 5.0]))
check("TODO 1b mse", lambda: math.isclose(mse(np.array([1.0, 2.0]), np.array([1.0, 4.0])), 2.0))

## A2 · Die Lösung aus der Statistik: Normalgleichungen

Für die lineare Regression gibt es eine geschlossene Lösung. Mit der Designmatrix $X = [x \;\; 1]$ (eine Spalte für $x$, eine Spalte Einsen für $b$) und $\theta = (w, b)^\top$ minimiert

$$\theta^* = (X^\top X)^{-1} X^\top y$$

den Loss. In NumPy löst man das Gleichungssystem $X^\top X\,\theta = X^\top y$ mit `np.linalg.solve` (stabiler als die Inverse zu bilden).

### TODO 2 · Kleinste Quadrate

In [ ]:
X_design = np.column_stack([x, np.ones_like(x)])    # Form (40, 2)

# TODO 2: theta_opt mit np.linalg.solve berechnen, dann w_opt und b_opt auslesen
theta_opt = None
w_opt, b_opt = (theta_opt if theta_opt is not None else (None, None))

print("w* =", w_opt, "| b* =", b_opt)

In [ ]:
check("TODO 2 Normalgleichungen", lambda: np.allclose([w_opt, b_opt], np.polyfit(x, y, 1)))
print("Minimaler Loss:", mse(y, vorhersage(x, w_opt, b_opt)) if theta_opt is not None else None)

> **Warum reicht das nicht?** Die geschlossene Lösung gibt es nur, weil der Loss quadratisch in $\theta$ ist. Bei neuronalen Netzen mit Millionen Parametern und Nichtlinearitäten gibt es keine Formel. Dort hilft nur ein iteratives Verfahren.

## A3 · Gradient Descent

Idee: Der **Gradient** $\nabla L$ zeigt in die Richtung des steilsten Anstiegs. Also gehen wir einen kleinen Schritt in die Gegenrichtung, immer wieder:

$$\theta \leftarrow \theta - \eta \, \nabla L(\theta)$$

$\eta$ ist die **Lernrate**. Für die MSE liefert die Kettenregel:

$$\frac{\partial L}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n} (y_i - \hat y_i)\,x_i \qquad \frac{\partial L}{\partial b} = -\frac{2}{n}\sum_{i=1}^{n} (y_i - \hat y_i)$$

### TODO 3 · Gradienten berechnen
Schreibt `gradienten(x, y, w, b)`, die das Paar `(dL_dw, dL_db)` zurückgibt. Der Test vergleicht mit einem **numerischen Gradienten** (Differenzenquotient). So prüft man jede Herleitung.

In [ ]:
def gradienten(x, y, w, b):
    # TODO 3: Residuen r = y - y_dach berechnen, dann beide Formeln oben umsetzen
    pass

In [ ]:
def numerischer_gradient(x, y, w, b, h=1e-6):
    """Zentraler Differenzenquotient: (L(θ+h) - L(θ-h)) / 2h für jeden Parameter."""
    gw = (mse(y, vorhersage(x, w + h, b)) - mse(y, vorhersage(x, w - h, b))) / (2 * h)
    gb = (mse(y, vorhersage(x, w, b + h)) - mse(y, vorhersage(x, w, b - h))) / (2 * h)
    return gw, gb

check("TODO 3 Gradient stimmt mit Differenzenquotient überein",
      lambda: np.allclose(gradienten(x, y, 0.5, 0.5), numerischer_gradient(x, y, 0.5, 0.5), rtol=1e-5))

### TODO 4 · Die Trainingsschleife
Startet bei $w = b = 0$ und führt 500 Schritte mit Lernrate $\eta = 0.1$ aus. Speichert den Loss jedes Schritts in `verlauf`.

In [ ]:
def gradient_descent(x, y, lr=0.1, schritte=500):
    w, b = 0.0, 0.0
    verlauf = []
    for _ in range(schritte):
        verlauf.append(mse(y, vorhersage(x, w, b)))
        # TODO 4: Gradienten holen und w, b einen Schritt gegen den Gradienten verschieben
        pass
    return w, b, verlauf

w_gd, b_gd, verlauf = gradient_descent(x, y)
print(f"Gradient Descent: w = {w_gd:.4f}, b = {b_gd:.4f}")
print(f"Normalgleichungen: w = {w_opt}, b = {b_opt}")

In [ ]:
check("TODO 4 Gradient Descent findet das Optimum", lambda: np.allclose([w_gd, b_gd], [w_opt, b_opt], atol=1e-4))
check("TODO 4 Loss sinkt", lambda: verlauf[-1] < verlauf[0])

## A4 · Die Lernrate entscheidet

Führt die Zelle aus: dreimal Gradient Descent mit verschiedenen Lernraten, jeweils 50 Schritte. Die y-Achse ist logarithmisch.

In [ ]:
plt.figure(figsize=(7, 3.5))
for lr in [0.01, 0.1, 0.2]:
    _, _, v = gradient_descent(x, y, lr=lr, schritte=50)
    plt.plot(v, label=f"Lernrate {lr}")
plt.yscale("log"); plt.xlabel("Schritt"); plt.ylabel("Loss (MSE)"); plt.legend(); plt.title("Zu klein, passend, zu groß")
plt.show()

> **Zum Nachdenken:** Bei $\eta = 0.01$ geht es langsam bergab, bei $0.1$ schnell, bei $0.2$ explodiert der Loss. Warum kann ein *zu großer* Schritt den Loss erhöhen? (Stellt euch eine Parabel vor, bei der ihr über das Tal hinausspringt.)

---
# Teil B · Dasselbe mit PyTorch

PyTorch kann **Ableitungen automatisch berechnen** (Autograd). Ihr schreibt nur den Loss auf, `loss.backward()` wendet die Kettenregel rückwärts durch alle Rechenschritte an. Das Ergebnis landet in `.grad`.

### TODO 5 · Autograd prüft eure Herleitung
Berechnet `loss_t` (die MSE) mit den PyTorch-Tensoren an der Stelle $w = b = 0.5$ und ruft `loss_t.backward()` auf.

In [ ]:
x_t = torch.tensor(x, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)
w_t = torch.tensor(0.5, requires_grad=True)     # requires_grad: PyTorch soll Ableitungen nach w merken
b_t = torch.tensor(0.5, requires_grad=True)

# TODO 5: Loss berechnen (wie mse, nur mit Tensoren) und backward() aufrufen
loss_t = None

if w_t.grad is not None:
    print("Autograd:  dL/dw =", w_t.grad.item(), "| dL/db =", b_t.grad.item())
print("Von Hand:  ", gradienten(x, y, 0.5, 0.5))

In [ ]:
check("TODO 5 Autograd = eure Herleitung", lambda: np.allclose([w_t.grad.item(), b_t.grad.item()], gradienten(x, y, 0.5, 0.5), rtol=1e-4))

Die komplette Regression in PyTorch, mit einem fertigen Optimierer (`SGD` = Gradient Descent). Diese fünf Zeilen im Schleifenrumpf sind das Grundgerüst **jedes** Trainings, auch bei großen Sprachmodellen.

In [ ]:
w_t = torch.zeros(1, requires_grad=True)
b_t = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([w_t, b_t], lr=0.1)

for schritt in range(500):
    optimizer.zero_grad()                              # 1. alte Gradienten löschen
    loss_t = ((y_t - (w_t * x_t + b_t)) ** 2).mean()   # 2. Vorhersage und Loss
    loss_t.backward()                                  # 3. Gradienten per Kettenregel
    optimizer.step()                                   # 4. Parameter-Update θ ← θ − η∇L

print(f"PyTorch: w = {w_t.item():.4f}, b = {b_t.item():.4f}")

---
# Teil C · Ein neuronales Netz klassifiziert Störungen

## C1 · Von Zahlen zu Wahrscheinlichkeiten

Bei **Klassifikation** gibt das Modell für jede Klasse eine Zahl aus, die **Logits** $z_k$. Die **Softmax** macht daraus Wahrscheinlichkeiten:

$$p_k = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

Der passende Loss ist die **Cross-Entropy**: Ist $k$ die richtige Klasse, dann ist $L = -\log p_k$. Sicher und richtig ergibt einen Loss nahe 0, sicher und falsch einen sehr großen.
Wer Statistik gehört hat, erkennt die negative Log-Likelihood: Cross-Entropy minimieren heißt Likelihood maximieren.

### TODO 6 · Softmax und Cross-Entropy
Implementiert beide Funktionen für **einen** Logit-Vektor. Trick für numerische Stabilität: Zieht vor dem `np.exp` das Maximum von `z` ab. Das ändert das Ergebnis nicht (warum?), verhindert aber einen Überlauf.

In [ ]:
def softmax(z):
    # TODO 6a
    pass

def cross_entropy(z, k):
    # TODO 6b: k ist der Index der richtigen Klasse
    pass

z_beispiel = np.array([2.0, 1.0, 0.1])
print("softmax:", softmax(z_beispiel))
print("Cross-Entropy, richtige Klasse 0:", cross_entropy(z_beispiel, 0))

In [ ]:
check("TODO 6a Summe 1", lambda: math.isclose(softmax(z_beispiel).sum(), 1.0))
check("TODO 6a gleiche Logits → gleichverteilt", lambda: np.allclose(softmax(np.zeros(4)), 0.25))
check("TODO 6a stabil bei großen Zahlen", lambda: np.all(np.isfinite(softmax(np.array([1000.0, 1001.0])))))
check("TODO 6b Cross-Entropy", lambda: math.isclose(cross_entropy(z_beispiel, 0), 0.41703, rel_tol=1e-4))

In [ ]:
# PyTorch rechnet dasselbe (Logits als Matrix: eine Zeile pro Beispiel)
print("PyTorch zum Vergleich:", F.cross_entropy(torch.tensor([[2.0, 1.0, 0.1]]), torch.tensor([0])).item())

## C2 · Die Daten: Sensorwerte bei Störungen

Bei jeder Störung zeichnet die Maschine vier Sensorwerte auf: **Spindeltemperatur** (°C), **Schwingstärke** (mm/s), **Motorstrom** (A) und **Kühlmitteldruck** (bar).
Ziel: Aus diesen vier Werten die Störungsart vorhersagen: `mechanisch`, `elektrisch` oder `Kühlung`.

Wir haben nur **45 gelabelte Trainingsfälle** und 600 weitere Fälle zur **Validierung**, also zum Prüfen auf Daten, die das Modell nie gesehen hat. Die Merkmale werden standardisiert (Mittelwert 0, Standardabweichung 1), damit alle auf ähnlicher Skala liegen.

In [ ]:
KLASSEN = ["mechanisch", "elektrisch", "Kühlung"]
MERKMALE = ["Temperatur °C", "Schwingung mm/s", "Strom A", "Druck bar"]
rng = np.random.default_rng(7)
mu = np.array([[62, 7.5, 14.0, 5.5], [58, 4.5, 18.0, 5.6], [70, 4.8, 14.5, 4.4]])
sd = np.array([[6, 2.2, 2.0, 0.8], [6, 2.0, 2.5, 0.8], [6, 2.0, 2.0, 0.9]])
def erzeuge(n):
    klasse = rng.integers(0, 3, n)
    return mu[klasse] + sd[klasse] * rng.standard_normal((n, 4)), klasse
X_train, y_train = erzeuge(45)
X_val, y_val = erzeuge(600)

m, s = X_train.mean(axis=0), X_train.std(axis=0)            # nur aus den Trainingsdaten!
Xtr_t = torch.tensor((X_train - m) / s, dtype=torch.float32)
Xva_t = torch.tensor((X_val - m) / s, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.long)
yva_t = torch.tensor(y_val, dtype=torch.long)
print("Training:", Xtr_t.shape, "| Validierung:", Xva_t.shape)

## C3 · Ein neuronales Netz bauen

Ein **neuronales Netz** ist eine Verkettung aus linearen Abbildungen und Nichtlinearitäten:

$$h_1 = \mathrm{ReLU}(W_1 x + b_1), \quad h_2 = \mathrm{ReLU}(W_2 h_1 + b_2), \quad z = W_3 h_2 + b_3$$

mit $\mathrm{ReLU}(a) = \max(0, a)$. Ohne die Nichtlinearität wäre die Verkettung wieder nur eine einzige lineare Abbildung.
In PyTorch stapelt man die Bausteine mit `nn.Sequential`: `nn.Linear(ein, aus)` ist $Wx + b$, `nn.ReLU()` die Nichtlinearität.

### TODO 7 · Netz definieren
`neues_modell(hidden)` soll ein Netz zurückgeben: 4 Eingänge → `hidden` Neuronen → ReLU → `hidden` Neuronen → ReLU → 3 Ausgänge (die Logits).

In [ ]:
def neues_modell(hidden=64):
    # TODO 7: return nn.Sequential(...)
    pass

In [ ]:
check("TODO 7a Ausgabeform", lambda: neues_modell()(Xtr_t).shape == (45, 3))
check("TODO 7b Anzahl Parameter", lambda: sum(p.numel() for p in neues_modell().parameters()) == 4675)

### TODO 8 · Der Trainingsschritt
Ergänzt im Schleifenrumpf die vier Zeilen aus Teil B: Gradienten löschen, Logits und Loss berechnen, `backward()`, `optimizer.step()`.
`nn.CrossEntropyLoss` erwartet die rohen Logits und wendet Softmax selbst an.

In [ ]:
def trainiere(modell, epochen=500, lr=0.003, weight_decay=0.0):
    optimizer = torch.optim.Adam(modell.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    verlauf = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for epoche in range(epochen):
        modell.train()
        # TODO 8: die vier Zeilen des Trainingsschritts (Daten: Xtr_t, ytr_t)
        pass

        modell.eval()
        with torch.no_grad():                               # Auswertung ohne Gradienten
            z_tr, z_va = modell(Xtr_t), modell(Xva_t)
            verlauf["train_loss"].append(loss_fn(z_tr, ytr_t).item())
            verlauf["val_loss"].append(loss_fn(z_va, yva_t).item())
            verlauf["train_acc"].append((z_tr.argmax(dim=1) == ytr_t).float().mean().item())
            verlauf["val_acc"].append((z_va.argmax(dim=1) == yva_t).float().mean().item())
    return verlauf

def zeige(verlauf, titel):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(verlauf["train_loss"], label="Training"); ax[0].plot(verlauf["val_loss"], label="Validierung")
    ax[0].set_title(f"Loss · {titel}"); ax[0].set_xlabel("Epoche"); ax[0].legend()
    ax[1].plot(verlauf["train_acc"], label="Training"); ax[1].plot(verlauf["val_acc"], label="Validierung")
    ax[1].set_title(f"Genauigkeit · {titel}"); ax[1].set_xlabel("Epoche"); ax[1].legend()
    plt.show()

In [ ]:
torch.manual_seed(0)
if neues_modell() is None:
    print("⏳ Erst TODO 7 lösen.")
else:
    v_gross = trainiere(neues_modell(64))
    zeige(v_gross, "großes Netz, 500 Epochen")
    beste = int(np.argmin(v_gross["val_loss"]))
    print(f"Ende:  Training {v_gross['train_acc'][-1]:.0%} | Validierung {v_gross['val_acc'][-1]:.0%}")
    print(f"Bester Validierungs-Loss in Epoche {beste}: Validierung {v_gross['val_acc'][beste]:.0%}")

In [ ]:
check("TODO 8 Netz lernt die Trainingsdaten", lambda: v_gross["train_acc"][-1] >= 0.95)
check("TODO 8 Loss sinkt", lambda: v_gross["train_loss"][-1] < v_gross["train_loss"][0])

> **Overfitting:** Die Trainingsgenauigkeit erreicht fast 100 %, die Validierung bleibt deutlich darunter. Der Validierungs-Loss sinkt erst und steigt dann wieder: Das Netz lernt die 45 Trainingsfälle **auswendig**, inklusive ihres Rauschens. Etwa 4.700 Parameter für 45 Beispiele sind viel zu viele.

## C4 · Gegenmittel

Führt die Zelle aus. **Weight Decay** bestraft große Gewichte und hält das Modell einfacher (Regularisierung).

In [ ]:
torch.manual_seed(0)
if neues_modell() is None:
    print("⏳ Erst TODO 7 und 8 lösen.")
else:
    v_wd = trainiere(neues_modell(64), weight_decay=0.05)
    zeige(v_wd, "mit Weight Decay")
    print(f"Ohne Weight Decay: Validierung {v_gross['val_acc'][-1]:.0%} | mit Weight Decay: {v_wd['val_acc'][-1]:.0%}")

Weitere Gegenmittel: **Early Stopping** (aufhören, wenn der Validierungs-Loss steigt), ein **kleineres Netz** und vor allem **mehr Daten**.
Wichtig: Wer anhand der Validierungsdaten Entscheidungen trifft, braucht für eine ehrliche Endbewertung noch einen dritten, unberührten **Testdatensatz**. Darauf kommen wir bei der Evaluation in Woche 9 zurück.

---
# Teil D · Ausblick: Ein Sprachmodell ist ein Klassifikator

Ein LLM macht im Kern dasselbe wie euer Netz: Es berechnet Logits und eine Softmax, nur über alle **Tokens** seines Wortschatzes (bei GPT-2 sind es 50.257). Trainiert wird mit Cross-Entropy auf das jeweils nächste Token.

In [ ]:
kontext = "Die Spindel läuft"
vokabular = ["unrund", "heiß", "rund", "leise", "Banane"]
logits = np.array([3.1, 1.9, 1.2, 0.4, -2.0])     # erfundene Werte, nur zur Illustration

if softmax(logits) is None:
    print("⏳ Erst TODO 6 lösen.")
else:
    p = softmax(logits)
    plt.figure(figsize=(6, 3))
    plt.bar(vokabular, p)
    plt.title(f"„{kontext} …“ – Wahrscheinlichkeit des nächsten Wortes")
    plt.show()
    for wort, wahrscheinlichkeit in zip(vokabular, p):
        print(f"{wort:8s} {wahrscheinlichkeit:.1%}")

## Bonus

**B1 · Eigene Experimente.** Variiert `hidden` (z. B. 4, 8, 64), `epochen`, `lr` und `weight_decay`. Welche Kombination erreicht die beste Validierungsgenauigkeit? Notiert euren Bestwert, wir vergleichen in der Auflösung.

**B2 · Lineares Modell als Vergleich.** Ersetzt das Netz durch `nn.Linear(4, 3)` (multinomiale logistische Regression). Wie schneidet es im Vergleich ab, und was sagt das über diese Daten?

In [ ]:
# B1 / B2

## Zusammenfassung

- Ein **Modell** ist eine Funktion $f_\theta$ mit Parametern. **Lernen** heißt, $\theta$ so zu wählen, dass ein **Loss** klein wird.
- Für lineare Regression gibt es eine geschlossene Lösung. Allgemein hilft nur **Gradient Descent**: $\theta \leftarrow \theta - \eta\nabla L$. Die **Lernrate** entscheidet über Konvergenz.
- Ein **neuronales Netz** verkettet lineare Abbildungen und Nichtlinearitäten. **Backpropagation** ist die Kettenregel, PyTorch erledigt sie mit `backward()`.
- Klassifikation nutzt **Softmax** und **Cross-Entropy** (negative Log-Likelihood).
- **Overfitting** erkennt man an der Lücke zwischen Training und Validierung.

### Ausblick auf Woche 4
Ein Sprachmodell sagt das nächste **Token** voraus. Aber was genau ist ein Token, und wie wird aus Text überhaupt ein Vektor, mit dem ein Netz rechnen kann?